In [1]:
import os
import sys
import zarr
import numpy as np
import pandas as pd
import xarray as xr
from glob import glob

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
def uv_to_spd(ds):
    ds['WRF_SPD10'] = np.sqrt(ds['WRF_U10']**2 + ds['WRF_V10']**2)
    ds = ds.drop_vars(('WRF_U10', 'WRF_V10'))
    return ds

year = 2020

ds_static = xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/static/C404_TC_static_8km.zarr')
lat_2d = ds_static['XLAT'].values
lon_2d = ds_static['XLONG'].values
ds_static['LANDMASK'].values # land>0; ocean=0

# C404 target
fn_target = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/C404_CorrDiff/TC_target_{year}.zarr'
ds_target = xr.open_zarr(fn_target)
ds_target = uv_to_spd(ds_target)
ds_target['WRF_PWAT'] = ds_target['WRF_PWAT_05']**2
ds_target['WRF_precip'] = ds_target['WRF_precip_025']**4
ds_target = ds_target.drop_vars(('WRF_PWAT_05', 'WRF_precip_025', 'WRF_Q_tot_05', 'WRF_T'))

# UNET baseline result
fn_unet = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/TC_UNET/TC_UNET_pred_{year}_MSLP.zarr'
ds_unet = xr.open_zarr(fn_unet)
ds_unet = uv_to_spd(ds_unet)

# ERA5-based corrdiff
fn_era5 = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/TC_pred_corrdiff_final/TC_ERA5_corrdiff_pred_{year}_mem*.zarr'
list_fn_era5 = sorted(glob(fn_era5))

list_ds_era5 = []
for fn_ in list_fn_era5:
    list_ds_era5.append(xr.open_zarr(fn_).isel(time=slice(100, 102)))
ds_era5 = xr.concat(list_ds_era5, dim='member')
ds_era5 = uv_to_spd(ds_era5)

# GDAS-based corrdiff
fn_gdas = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/TC_pred_corrdiff_final/TC_GDAS_corrdiff_pred_{year}_mem*.zarr'
list_fn_gdas = sorted(glob(fn_gdas))

list_ds_gdas = []
for fn_ in list_fn_gdas:
    list_ds_gdas.append(xr.open_zarr(fn_).isel(time=slice(100, 102)))
ds_gdas = xr.concat(list_ds_gdas, dim='member')
ds_gdas = uv_to_spd(ds_gdas)

varnames = ['WRF_MSLP', 'WRF_PWAT', 'WRF_SP', 'WRF_T2', 'WRF_precip', 'WRF_SPD10']

# Verification content
# 1. Core scores
# 2. Calibration, stratified by regime
# 3. Spectra